In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../northbeam_feature_engineered.csv")

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
contract_value,424.0,91926.470472,215886.949785,-4500.000000,3544.005000,9726.250000,72607.442500,1.322116e+06
seats_licensed,424.0,201.580189,327.647670,5.000000,25.000000,68.000000,200.250000,1.488000e+03
seats_active,424.0,135.537736,242.502884,2.000000,17.000000,40.000000,127.000000,1.301000e+03
mau_m5,424.0,82.992925,169.800766,0.000000,5.000000,18.500000,74.250000,1.206000e+03
mau_m4,424.0,81.372642,164.123873,0.000000,5.000000,18.000000,69.250000,1.151000e+03
mau_m3,424.0,77.608491,153.878978,0.000000,5.000000,19.000000,72.000000,1.147000e+03
mau_m2,424.0,71.823113,150.319060,0.000000,3.000000,16.000000,68.250000,1.080000e+03
mau_m1,424.0,69.214623,148.807915,0.000000,3.000000,15.500000,70.000000,1.069000e+03
mau_m0,424.0,65.181604,153.742770,0.000000,1.000000,13.000000,60.250000,1.154000e+03
support_tickets_90d,424.0,4.573113,2.629948,0.000000,3.000000,4.000000,6.000000,1.300000e+01


In [5]:
df["seat_utilization"] = df["seat_utilization"].clip(upper=100)

In [6]:
df["seat_utilization"].describe()

count    424.000000
mean      66.372767
std       18.325938
min       33.333333
25%       50.000000
50%       67.152322
75%       81.947439
max      100.000000
Name: seat_utilization, dtype: float64

In [7]:
def seat_score(utilization):

    if utilization >= 82:
        return 25

    elif utilization >= 67:
        return 20

    elif utilization >= 50:
        return 10

    else:
        return 0

In [8]:
df["seat_score"] = df["seat_utilization"].apply(seat_score)

df[["seat_utilization","seat_score"]].head()

,seat_utilization,seat_score
0,39.534884,0
1,74.305556,20
2,46.153846,0
3,75.000000,20
4,100.000000,25


In [9]:
median_csat = df["csat"].median()

df["csat"] = df["csat"].fillna(median_csat)

In [10]:
df["csat"].isnull().sum()

np.int64(0)

In [11]:
def csat_score(csat):

    if csat >= 4.2:
        return 20

    elif csat >= 3.7:
        return 15

    elif csat >= 3.1:
        return 10

    else:
        return 0

In [12]:
df["csat_score"] = df["csat"].apply(csat_score)

df[["csat","csat_score"]].head()

,csat,csat_score
0,3.7,15
1,4.3,20
2,3.7,15
3,3.1,10
4,3.7,15


In [13]:
def mau_score(avg_mau):

    if avg_mau >= 68:
        return 20

    elif avg_mau >= 17:
        return 15

    elif avg_mau >= 5:
        return 10

    else:
        return 0

In [14]:
df["mau_score"] = df["avg_mau"].apply(mau_score)

df[["avg_mau","mau_score"]].head()

,avg_mau,mau_score
0,0.833333,0
1,4.500000,0
2,10.500000,10
3,37.833333,15
4,5.166667,10


In [15]:
def usage_score(trend):

    if trend > 3:
        return 15

    elif trend >= 0:
        return 10

    elif trend >= -10:
        return 5

    else:
        return 0

In [16]:
df["usage_score"] = df["usage_trend"].apply(usage_score)

df[["usage_trend","usage_score"]].head()

,usage_trend,usage_score
0,1,10
1,2,10
2,0,10
3,-48,0
4,-7,5


In [17]:
def support_score(tickets):

    if tickets <= 3:
        return 10

    elif tickets <= 6:
        return 7

    elif tickets <= 9:
        return 4

    else:
        return 0

In [18]:
df["support_score"] = df["support_tickets_90d"].apply(support_score)

df[["support_tickets_90d","support_score"]].head()

,support_tickets_90d,support_score
0,6,7
1,0,10
2,6,7
3,7,4
4,1,10


In [19]:
def escalation_score(escalations):

    if escalations == 0:
        return 10

    elif escalations == 1:
        return 7

    elif escalations == 2:
        return 4

    else:
        return 0

In [20]:
df["escalation_score"] = df["escalations_90d"].apply(escalation_score)

df[["escalations_90d","escalation_score"]].head()

,escalations_90d,escalation_score
0,0,10
1,0,10
2,1,7
3,0,10
4,0,10


In [21]:
df["customer_health_score"] = (

      df["seat_score"]
    + df["mau_score"]
    + df["csat_score"]
    + df["usage_score"]
    + df["support_score"]
    + df["escalation_score"]

)

In [22]:
df[[
    "seat_score",
    "mau_score",
    "csat_score",
    "usage_score",
    "support_score",
    "escalation_score",
    "customer_health_score"
]].head()

,seat_score,mau_score,csat_score,usage_score,support_score,escalation_score,customer_health_score
0,0,0,15,10,7,10,42
1,20,0,20,10,10,10,70
2,0,10,15,10,7,7,49
3,20,15,10,0,4,10,59
4,25,10,15,5,10,10,75


In [23]:
def customer_status(score):

    if score >= 80:
        return "Healthy"

    elif score >= 50:
        return "Monitor"

    else:
        return "At Risk"

In [24]:
df["customer_status"] = df["customer_health_score"].apply(customer_status)

df[["customer_health_score","customer_status"]].head()

,customer_health_score,customer_status
0,42,At Risk
1,70,Monitor
2,49,At Risk
3,59,Monitor
4,75,Monitor


In [25]:
df = df.sort_values(
    by="customer_health_score",
    ascending=True
)

df["priority_rank"] = range(1, len(df)+1)

In [26]:
df[[
    "account_name",
    "customer_health_score",
    "customer_status",
    "priority_rank"
]].head(10)

,account_name,customer_health_score,customer_status,priority_rank
186,Yarrow Group,19,At Risk,1
307,Vista Partners,22,At Risk,2
244,Helix Partners,24,At Risk,3
17,Larkfield Foods,24,At Risk,4
192,Pinnacle Media,25,At Risk,5
380,Juniper Foods,26,At Risk,6
97,Quantum Supply,26,At Risk,7
206,Maple Dynamics,28,At Risk,8
411,Garnet Group,29,At Risk,9
355,Crestline Health,29,At Risk,10


In [27]:
top_40 = df.head(40)

In [28]:
top_40

,account_id,account_name,segment,industry,region,signup_date,contract_start_date,contract_end_date,billing_term,contract_value,...,has_escalation,seat_score,csat_score,mau_score,usage_score,support_score,escalation_score,customer_health_score,customer_status,priority_rank
186,ACC-1091,Yarrow Group,SMB,Manufacturing,EMEA,2023-05-24,2023-05-24,2026-08-19,monthly,476.33,...,1,0,0,0,5,7,7,19,At Risk,1
307,ACC-1169,Vista Partners,Mid-Market,Education,North America,2023-01-07,2023-01-07,2027-01-03,annual,73368.19,...,1,0,0,15,0,7,0,22,At Risk,2
244,ACC-1027,Helix Partners,SMB,Construction,North America,2024-11-19,2024-11-19,2026-08-05,monthly,614.96,...,1,0,0,0,10,10,4,24,At Risk,3
17,ACC-1235,Larkfield Foods,Mid-Market,FinServ,EMEA,2026-03-21,2026-03-21,2026-04-04,monthly,3326.91,...,1,0,0,10,0,7,7,24,At Risk,4
192,ACC-1344,Pinnacle Media,SMB,SaaS,North America,2025-11-06,2025-11-06,2026-11-10,annual,4769.00,...,0,0,0,0,5,10,10,25,At Risk,5
380,ACC-1195,Juniper Foods,SMB,Hospitality,APAC,2024-11-22,2024-11-22,2026-07-27,annual,2389.31,...,1,10,0,0,5,4,7,26,At Risk,6
97,ACC-1157,Quantum Supply,SMB,Healthcare,APAC,2022-11-11,2022-11-11,2026-08-28,monthly,362.79,...,1,0,10,0,5,7,4,26,At Risk,7
206,ACC-1297,Maple Dynamics,Mid-Market,FinServ,North America,2025-02-23,2025-02-23,2026-10-06,annual,49384.31,...,1,0,15,0,5,4,4,28,At Risk,8
411,ACC-1345,Garnet Group,SMB,Hospitality,EMEA,2026-04-20,2026-04-20,2026-12-09,annual,4536.69,...,1,10,0,0,5,7,7,29,At Risk,9
355,ACC-1397,Crestline Health,Mid-Market,FinServ,North America,2022-11-02,2022-11-02,2025-09-20,monthly,6921.84,...,1,0,0,10,5,10,4,29,At Risk,10


In [30]:
# Complete dataset with health scores
df.to_csv("../northbeam_customer_health_scores.csv", index=False)

top_40.to_csv("../top_40_priority_customers.csv", index=False)